<a href="https://colab.research.google.com/github/Chimatanagagopal/DEEP-LEARNING_practice/blob/main/rnn_own_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

long_reviews = [
    "the weather was nice today and we went outside and played games and had food and the movie was fantastic loved it",
    "we traveled far and saw many things and met people and had adventures and visited places and the film was amazing wonderful",
    "spent time reading books and learning things and practicing skills and working hard and studying well and movie was brilliant loved",
    "days passed slowly and nothing happened and time went by and seasons changed and years flew and eventually movie was excellent fantastic",
    "morning started dull and afternoon was boring and evening came slowly and night arrived late and finally the film was great loved",
    "the weather was bad today and we stayed inside and did nothing and ate food and the movie was terrible hated it",
    "we stayed home and saw nothing and met nobody and had no adventures and visited nowhere and the film was awful hated",
    "spent time doing nothing and learning nothing and wasting time and working slow and studying badly and movie was worst hated completely",
    "days passed quickly and everything happened wrong and time was wasted and seasons were bad and movie was boring hated it completely",
    "morning started wrong and afternoon was terrible and evening came badly and night arrived early and finally the film was bad hated",
]

long_labels = [1,1,1,1,1, 0,0,0,0,0]

word_set = set()
for review in long_reviews:
    for word in review.split():
        word_set.add(word)

long_vocab = {"<PAD>": 0}
for i, word in enumerate(sorted(word_set)):
    long_vocab[word] = i + 1

def encode(review, vocab):
    return [vocab.get(w, 0) for w in review.split()]

X_long = [encode(r, long_vocab) for r in long_reviews]

lengths = [len(x) for x in X_long]
print("Review lengths:")
for i, l in enumerate(lengths):
    print(f"  Review {i+1}: {l} words")

MAX_LONG = max(lengths)
X_long_pad = pad_sequences(X_long, maxlen=MAX_LONG, padding="pre")
y_long     = np.array(long_labels)

print(f"\nMax length:    {MAX_LONG} words")
print(f"Dataset shape: {X_long_pad.shape}")
print(f"\nKey insight: sentiment word is ALWAYS at the END")
print(f"RNN must remember it across {MAX_LONG} steps!")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

VOCAB_SIZE_L = len(long_vocab) + 1
EMBED_DIM    = 16
RNN_UNITS    = 8

def build_rnn(vocab_size, embed_dim, rnn_units, max_len):
    model = Sequential([
        Embedding(input_dim=vocab_size,
                  output_dim=embed_dim,
                  input_length=max_len),
        SimpleRNN(units=rnn_units,
                  activation="tanh"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(learning_rate=0.01),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

print("Training SHORT sequence RNN (7 words)...")
model_short = build_rnn(len(word_to_idx)+1, 8, 4, MAX_LEN)
history_short = model_short.fit(
    X, y,
    epochs=50,
    batch_size=2,
    verbose=0
)
print(f"Short RNN final accuracy: {history_short.history['accuracy'][-1]*100:.1f}%")

print("\nTraining LONG sequence RNN (20+ words)...")
model_long = build_rnn(VOCAB_SIZE_L, EMBED_DIM, RNN_UNITS, MAX_LONG)
history_long = model_long.fit(
    X_long_pad, y_long,
    epochs=50,
    batch_size=2,
    verbose=0
)
print(f"Long RNN final accuracy:  {history_long.history['accuracy'][-1]*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_short.history["accuracy"],
             color="steelblue", linewidth=2,
             label=f"Short ({MAX_LEN} words)")
axes[0].plot(history_long.history["accuracy"],
             color="coral", linewidth=2,
             label=f"Long ({MAX_LONG} words)")
axes[0].set_title("Short vs Long Sequence Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_short.history["loss"],
             color="steelblue", linewidth=2,
             label=f"Short ({MAX_LEN} words)")
axes[1].plot(history_long.history["loss"],
             color="coral", linewidth=2,
             label=f"Long ({MAX_LONG} words)")
axes[1].set_title("Short vs Long Sequence Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Vanishing Gradient — Short vs Long Sequence", fontsize=14)
plt.tight_layout()
plt.show()